# Same Anwser Anomaly

In [1]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt

## Import Data from csv

In [2]:
fca_question = pd.read_csv("../../../decoded_data/FCA/FactQuestionFCA.csv")

## Data Inspection

In [3]:
fca_question.head()

,FCAQuestionKey,TestKey,Competence1Key,Competence2Key,Competence3Key,Competence4Key,ItemId,Answer1,Answer2,Answer3,TimeSpent
0,1,1,1,0,0,0,222,0,0,0,11
1,2,1,2,183,0,0,223,3,4,2,3
2,3,1,3,184,0,0,226,4,3,2,13
3,4,1,4,0,0,0,229,2,0,4,100
4,5,1,5,0,0,0,238,0,0,0,2


In [4]:
fca_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035705 entries, 0 to 3035704
Data columns (total 11 columns):
 #   Column          Dtype
---  ------          -----
 0   FCAQuestionKey  int64
 1   TestKey         int64
 2   Competence1Key  int64
 3   Competence2Key  int64
 4   Competence3Key  int64
 5   Competence4Key  int64
 6   ItemId          int64
 7   Answer1         int64
 8   Answer2         int64
 9   Answer3         int64
 10  TimeSpent       int64
dtypes: int64(11)
memory usage: 254.8 MB


## Data Preparation

The only data cleaning that needs to happen here is getting rid of the coloms we won't need in this anomaly detection. There seem to be no empty fields or suspicious values to worry about. 

In [ ]:
df = fca_question[["QuestionKey", "Answer1", "Answer2", "Answer3"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035705 entries, 0 to 3035704
Data columns (total 4 columns):
 #   Column          Dtype
---  ------          -----
 0   FCAQuestionKey  int64
 1   Answer1         int64
 2   Answer2         int64
 3   Answer3         int64
dtypes: int64(4)
memory usage: 92.6 MB


## Data Labelling
We will now figure out on which tests the candidate filled out exactly the same answer for every question. 

In [6]:
same_answers = (df['Answer1'] == df['Answer2']) & (df['Answer2'] == df['Answer3'])
df['SameAnswers'] = False
df.loc[same_answers, 'SameAnswers'] = True

C:\Users\monad\AppData\Local\Temp\ipykernel_14812\1524627831.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['SameAnswers'] = False


In [8]:
df.head()

,FCAQuestionKey,Answer1,Answer2,Answer3,SameAnswers
0,1,0,0,0,True
1,2,3,4,2,False
2,3,4,3,2,False
3,4,2,0,4,False
4,5,0,0,0,True


## Exporting Data with Anomaly Check

In [ ]:
df = df[["QuestionKey", "SameAnswers"]]
df.set_index("QuestionKey", inplace=True)
df

,SameAnswers
FCAQuestionKey,
1,True
2,False
3,False
4,False
5,True
...,...
3035701,False
3035702,False
3035703,False


In [10]:
df.to_csv("../csv/same_answers_question_checked.csv")